In [ ]:
# CELL 1 — Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

Libraries loaded!


In [ ]:
# CELL 2 — Load dataset
df_raw = pd.read_excel('/content/Dataset for Data Analytics.xlsx')
df= df_raw.copy()  # always keep the raw data untouched

print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

Dataset loaded: 1200 rows x 14 columns


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## Phase 1 — Data Audit

In [ ]:
# CELL 3 — Overview
print('='*55)
print('  DATA AUDIT REPORT')
print('='*55)
print(f'  Total rows    : {df.shape[0]}')
print(f'  Total columns : {df.shape[1]}')
print()
print('--- Column types ---')
print(df.dtypes)
print()
print('--- Missing values ---')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(audit[audit['Missing'] > 0])
print()
print('--- Duplicate rows ---')
print(f'  Full duplicates  : {df.duplicated().sum()}')
print(f'  Duplicate OrderID: {df["OrderID"].duplicated().sum()}')

  DATA AUDIT REPORT
  Total rows    : 1200
  Total columns : 14

--- Column types ---
OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

--- Missing values ---
            Missing  Percentage
CouponCode      309       25.75

--- Duplicate rows ---
  Full duplicates  : 0
  Duplicate OrderID: 0


In [ ]:
# CELL 4 — Statistical summary
print('--- Numeric columns summary ---')
print(df[['Quantity','UnitPrice','TotalPrice','ItemsInCart']].describe().round(2))
print()
print('--- Categorical columns unique values ---')
cat_cols = ['Product','OrderStatus','PaymentMethod','ReferralSource','CouponCode']
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} unique -> {df[col].dropna().unique().tolist()}')

--- Numeric columns summary ---
       Quantity  UnitPrice  TotalPrice  ItemsInCart
count   1200.00    1200.00     1200.00      1200.00
mean       2.95     356.41     1053.97         5.48
std        1.41     197.18      819.86         2.28
min        1.00      11.39       11.39         1.00
25%        2.00     186.06      410.52         4.00
50%        3.00     364.21      823.62         5.00
75%        4.00     521.57     1578.48         7.00
max        5.00     699.93     3456.40        10.00

--- Categorical columns unique values ---
  Product: 7 unique -> ['Monitor', 'Phone', 'Tablet', 'Chair', 'Printer', 'Laptop', 'Desk']
  OrderStatus: 5 unique -> ['Shipped', 'Cancelled', 'Returned', 'Delivered', 'Pending']
  PaymentMethod: 5 unique -> ['Debit Card', 'Online', 'Credit Card', 'Gift Card', 'Cash']
  ReferralSource: 5 unique -> ['Instagram', 'Referral', 'Email', 'Facebook', 'Google']
  CouponCode: 3 unique -> ['SAVE10', 'FREESHIP', 'WINTER15']


## Phase 2 — Strategic Imputation (Missing Values)

In [ ]:
# CELL 5 — Handle missing CouponCode
# CouponCode has 309 missing values (25.75%)
# Strategy: fill with 'NO_COUPON' — it means no coupon was used

missing_before = df['CouponCode'].isnull().sum()

df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

missing_after = df['CouponCode'].isnull().sum()

print('CouponCode imputation:')
print(f'  Missing before : {missing_before}')
print(f'  Missing after  : {missing_after}')
print(f'  Strategy       : fillna(NO_COUPON) — no coupon used')
print(f'  Records saved  : {missing_before}')
print()
print('CouponCode distribution after imputation:')
print(df['CouponCode'].value_counts())

CouponCode imputation:
  Missing before : 309
  Missing after  : 0
  Strategy       : fillna(NO_COUPON) — no coupon used
  Records saved  : 309

CouponCode distribution after imputation:
CouponCode
FREESHIP     313
NO_COUPON    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64


## Phase 3 — Integrity Audit (Duplicates)

In [ ]:
# CELL 6 — Check and remove duplicates
print('--- Before deduplication ---')
print(f'  Total rows           : {len(df)}')
print(f'  Full duplicate rows  : {df.duplicated().sum()}')
print(f'  Duplicate OrderIDs   : {df["OrderID"].duplicated().sum()}')

# Remove full duplicate rows
dupes_removed = df.duplicated().sum()
df = df.drop_duplicates()

print()
print('--- After deduplication ---')
print(f'  Total rows     : {len(df)}')
print(f'  Rows removed   : {dupes_removed}')
print(f'  All OrderIDs unique: {df["OrderID"].is_unique}')

--- Before deduplication ---
  Total rows           : 1200
  Full duplicate rows  : 0
  Duplicate OrderIDs   : 0

--- After deduplication ---
  Total rows     : 1200
  Rows removed   : 0
  All OrderIDs unique: True


## Phase 4 — Standardization (Formats)

In [ ]:
# CELL 7 — Fix Date format to ISO 8601 (YYYY-MM-DD)
print('Date before:', df['Date'].dtype)
print('Sample:', df['Date'].head(3).tolist())

# Convert to datetime then format as YYYY-MM-DD string
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

print()
print('Date after:', df['Date'].dtype)
print('Sample:', df['Date'].head(3).tolist())
print()
# Verify no bad dates
invalid_dates = pd.to_datetime(df['Date'], errors='coerce').isnull().sum()
print(f'Invalid dates: {invalid_dates}  <- must be 0')

Date before: datetime64[ns]
Sample: [Timestamp('2023-01-04 00:00:00'), Timestamp('2024-08-23 00:00:00'), Timestamp('2024-02-27 00:00:00')]

Date after: object
Sample: ['2023-01-04', '2024-08-23', '2024-02-27']

Invalid dates: 0  <- must be 0


In [ ]:
# CELL 8 — Fix text columns (strip whitespace + Proper Case)
text_cols = ['Product','OrderStatus','PaymentMethod','ReferralSource','ShippingAddress']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print('Text columns standardized (strip + title case):')
for col in text_cols:
    print(f'  {col}: {df[col].unique().tolist()}')

Text columns standardized (strip + title case):
  Product: ['Monitor', 'Phone', 'Tablet', 'Chair', 'Printer', 'Laptop', 'Desk']
  OrderStatus: ['Shipped', 'Cancelled', 'Returned', 'Delivered', 'Pending']
  PaymentMethod: ['Debit Card', 'Online', 'Credit Card', 'Gift Card', 'Cash']
  ReferralSource: ['Instagram', 'Referral', 'Email', 'Facebook', 'Google']
  ShippingAddress: ['928 Main St', '823 Main St', '512 Main St', '275 Main St', '668 Main St', '934 Main St', '986 Main St', '706 Main St', '904 Main St', '102 Main St', '333 Main St', '831 Main St', '179 Main St', '903 Main St', '980 Main St', '942 Main St', '914 Main St', '382 Main St', '891 Main St', '273 Main St', '587 Main St', '569 Main St', '633 Main St', '755 Main St', '956 Main St', '709 Main St', '296 Main St', '881 Main St', '568 Main St', '930 Main St', '895 Main St', '996 Main St', '830 Main St', '912 Main St', '926 Main St', '128 Main St', '465 Main St', '409 Main St', '549 Main St', '666 Main St', '267 Main St', '657 Mai

In [ ]:
# CELL 9 — Fix numeric precision (2 decimal places)
df['UnitPrice']  = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

print('Numeric precision fixed to 2 decimals:')
print(df[['UnitPrice','TotalPrice']].head(5))

# Verify TotalPrice = Quantity * UnitPrice
df['_check'] = (df['Quantity'] * df['UnitPrice']).round(2)
mismatches   = (df['TotalPrice'] != df['_check']).sum()
df.drop(columns=['_check'], inplace=True)
print(f'\nTotalPrice mismatches: {mismatches}  <- must be 0')

Numeric precision fixed to 2 decimals:
   UnitPrice  TotalPrice
0     570.62     2853.10
1     151.35      302.70
2     550.68     2753.40
3     273.19      273.19
4     626.01     2504.04

TotalPrice mismatches: 0  <- must be 0


## Phase 5 — Verification Gate (0% Error Rate)

In [ ]:
# CELL 10 — Verification Gate
print('='*55)
print('  VERIFICATION GATE')
print('='*55)

checks = {}

# Check 1: Zero missing values
total_missing = df.isnull().sum().sum()
checks['Missing values'] = total_missing
print(f'  Missing values       : {total_missing}  {"PASS" if total_missing == 0 else "FAIL"}')

# Check 2: Zero duplicate OrderIDs
dup_ids = df['OrderID'].duplicated().sum()
checks['Duplicate OrderIDs'] = dup_ids
print(f'  Duplicate OrderIDs   : {dup_ids}  {"PASS" if dup_ids == 0 else "FAIL"}')

# Check 3: Zero duplicate rows
dup_rows = df.duplicated().sum()
checks['Duplicate rows'] = dup_rows
print(f'  Duplicate rows       : {dup_rows}  {"PASS" if dup_rows == 0 else "FAIL"}')

# Check 4: Date format correct
bad_dates = pd.to_datetime(df['Date'], errors='coerce').isnull().sum()
checks['Bad date formats'] = bad_dates
print(f'  Bad date formats     : {bad_dates}  {"PASS" if bad_dates == 0 else "FAIL"}')

# Check 5: Negative values
neg_vals = (df[['Quantity','UnitPrice','TotalPrice']] < 0).sum().sum()
checks['Negative values'] = neg_vals
print(f'  Negative values      : {neg_vals}  {"PASS" if neg_vals == 0 else "FAIL"}')

print()
all_pass = all(v == 0 for v in checks.values())
print(f'  FINAL RESULT: {"ALL CHECKS PASSED - READY FOR PROJECT 2" if all_pass else "ISSUES FOUND - FIX BEFORE PROCEEDING"}')
print('='*55)

  VERIFICATION GATE
  Missing values       : 0  PASS
  Duplicate OrderIDs   : 0  PASS
  Duplicate rows       : 0  PASS
  Bad date formats     : 0  PASS
  Negative values      : 0  PASS

  FINAL RESULT: ALL CHECKS PASSED - READY FOR PROJECT 2


## Phase 6 — Change Log

In [ ]:
# CELL 11 — Change Log (required by DecodeLabs)
change_log = pd.DataFrame([
    {
        'Change ID':   'CR001',
        'Column':      'CouponCode',
        'Issue':       'Missing values (309 rows = 25.75%)',
        'Strategy':    'fillna(NO_COUPON) — no coupon used',
        'Impact':      'Preserved 309 records',
        'Status':      'Resolved'
    },
    {
        'Change ID':   'CR002',
        'Column':      'All rows',
        'Issue':       'Duplicate rows check',
        'Strategy':    'drop_duplicates()',
        'Impact':      'Verified 0 duplicates — no rows removed',
        'Status':      'Resolved'
    },
    {
        'Change ID':   'CR003',
        'Column':      'Date',
        'Issue':       'Format standardization',
        'Strategy':    'Convert to ISO 8601 (YYYY-MM-DD)',
        'Impact':      '1200 dates standardized',
        'Status':      'Resolved'
    },
    {
        'Change ID':   'CR004',
        'Column':      'Text columns',
        'Issue':       'Inconsistent casing and whitespace',
        'Strategy':    'str.strip().str.title()',
        'Impact':      '5 columns standardized',
        'Status':      'Resolved'
    },
    {
        'Change ID':   'CR005',
        'Column':      'UnitPrice, TotalPrice',
        'Issue':       'Numeric precision',
        'Strategy':    'round(2) — 2 decimal places',
        'Impact':      '2 columns standardized',
        'Status':      'Resolved'
    },
])

print('CHANGE LOG — DecodeLabs Project 1')
print('='*80)
print(change_log.to_string(index=False))

CHANGE LOG — DecodeLabs Project 1
Change ID                Column                              Issue                           Strategy                                  Impact   Status
    CR001            CouponCode Missing values (309 rows = 25.75%) fillna(NO_COUPON) — no coupon used                   Preserved 309 records Resolved
    CR002              All rows               Duplicate rows check                  drop_duplicates() Verified 0 duplicates — no rows removed Resolved
    CR003                  Date             Format standardization   Convert to ISO 8601 (YYYY-MM-DD)                 1200 dates standardized Resolved
    CR004          Text columns Inconsistent casing and whitespace            str.strip().str.title()                  5 columns standardized Resolved
    CR005 UnitPrice, TotalPrice                  Numeric precision        round(2) — 2 decimal places                  2 columns standardized Resolved


In [ ]:
# CELL 12 — Export cleaned dataset
df.to_excel('Dataset_Cleaned.xlsx', index=False)
df.to_csv('Dataset_Cleaned.csv',   index=False)

change_log.to_excel('Change_Log.xlsx', index=False)

print('Files exported:')
print('  Dataset_Cleaned.xlsx')
print('  Dataset_Cleaned.csv')
print('  Change_Log.xlsx')
print()
print(f'Final dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print('--- Final dataset preview ---')
df.head()

Files exported:
  Dataset_Cleaned.xlsx
  Dataset_Cleaned.csv
  Change_Log.xlsx

Final dataset: 1200 rows x 14 columns

--- Final dataset preview ---


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
